<a href="https://colab.research.google.com/github/komazawa-deep-learning/komazawa-deep-learning.github.io/blob/master/2026notebooks/2026psy3a_lect07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IAM モデルのデモ


In [ ]:
try:
    import japanize_matplotlib
except ImportError:
    !pip install japanize_matplotlib
    import japanize_matplotlib

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def simulate_word_superiority(condition, cycles=50):
    # --- 1. ネットワークの初期化 ---
    # 文字層 (5ノード): [ T(1文字目), D(1文字目), X(1文字目), O(2文字目), X(2文字目) ]
    letters = np.zeros(5)
    # 単語層 (2ノード): [ "TO", "DO" ]
    words = np.zeros(2)

    # --- 2. 重み行列の定義 (アーキテクチャの配線) ---
    # 【ボトムアップ重み】 (文字層 -> 単語層) : 5行2列
    W_L2W = np.array([
        [ 1.0, -0.2], # T(1) は TO を興奮させ、DO を抑制する
        [-0.2,  1.0], # D(1) は DO を興奮させ、TO を抑制する
        [-0.5, -0.5], # X(1) は どの単語の構成要素でもないため両方を抑制
        [ 1.0,  1.0], # O(2) は TO と DO の両方を興奮させる
        [-0.5, -0.5]  # X(2) は 両方を抑制
    ])

    # 【トップダウン重み】 (単語層 -> 文字層) : 2行5列
    W_W2L = np.array([
        [ 1.0,  0.0,  0.0,  1.0,  0.0], # "TO" が発火すれば、T(1) と O(2) にフィードバックを送る
        [ 0.0,  1.0,  0.0,  1.0,  0.0]  # "DO" が発火すれば、D(1) と O(2) にフィードバックを送る
    ])

    # 【単語層内の側抑制】 (層内の競合) : 2行2列
    W_W2W_inh = np.array([
        [ 0.0, -0.5],
        [-0.5,  0.0]
    ])

    # --- 3. 入力刺激の設定 ---
    # 実験のターゲットは常に「2 文字目の 'O' (インデックス3)」の活性化速度とする。
    if condition == "word":
        # 条件A: 単語文脈 (刺激 "TO" を提示)
        ext_input = np.array([1.0, 0.0, 0.0, 1.0, 0.0])
    elif condition == "nonword":
        # 条件B: 非語文脈 (刺激 "XO" を提示)
        ext_input = np.array([0.0, 0.0, 1.0, 1.0, 0.0])
    else:
        raise ValueError("無効な条件です")

    # シミュレーションパラメータ
    tau = 0.1    # カスケード率（情報の伝播速度）
    decay = 0.05 # 減衰率（時間経過による活性化の低下）

    history_target_O = []

    # --- 4. 時間ステップごとの情報処理（ループ） ---
    for _ in range(cycles):
        # [Step A] 単語層の更新
        # (文字層からのボトムアップ入力) + (単語層内の側抑制)
        net_words = np.dot(letters, W_L2W) + np.dot(words, W_W2W_inh)
        words = np.maximum(0, words + tau * net_words - decay * words)

        # [Step B] 文字層の更新
        # (外部からの視覚入力) + (単語層からのトップダウン・フィードバック)
        net_letters = ext_input + np.dot(words, W_W2L)
        letters = np.maximum(0, letters + tau * net_letters - decay * letters)

        # ターゲット文字 'O(2)' の活性化量を記録
        history_target_O.append(letters)

    return history_target_O

# --- 実行と結果の可視化 ---
cycles = 50
history_word = simulate_word_superiority("word", cycles)
history_nonword = simulate_word_superiority("nonword", cycles)

plt.figure(figsize=(10, 6))
plt.plot(history_word, label="ターゲット O (単語 TO 内)", color="blue", linewidth=3)
plt.plot(history_nonword, label="ターゲット O (非単語 XO 内)", color="red", linestyle='--', linewidth=3)

plt.title("単語優位性効果のシミュレーション (IAM)", fontsize=14)
plt.xlabel("時間ステップ (処理時間)", fontsize=12)
plt.ylabel("ターゲット文字ノード O の活性値", fontsize=12)
plt.axhline(y=2.5, color='gray', linestyle=':', label="認知しきい値")
plt.legend(fontsize=12)
plt.grid(True)
plt.show()

# Dell モデル

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
import numpy as np
from typing import Dict, List, Tuple, Optional, Literal

JoltMode = Literal["lex", "phon", "both", "none"]


@dataclass(frozen=True)
class LexItem:
    word: str
    sem: np.ndarray
    phon: Tuple[int, ...]


@dataclass(frozen=True)
class Params:
    w_sem_lex: float = 1.0
    w_lex_sem: float = 1.0
    w_lex_phon: float = 1.0
    w_phon_lex: float = 1.0

    decay: float = 0.25
    gain: float = 1.0
    noise_sd: float = 0.05

    n_steps: int = 16
    jolt_step: int = 8

    jolt_mode: JoltMode = "lex"
    jolt_lex: float = 0.6
    jolt_phon: float = 0.4

    temperature_lex: float = 0.7
    temperature_phon: float = 0.7

    omission_thresh_lex: float = 0.05


@dataclass(frozen=True)
class TrialOut:
    target_word: str
    chosen_word: Optional[str]
    produced_phon: Optional[Tuple[int, ...]]
    error_type: str


class FoygelDell2000_SlotPhon:
    def __init__(
        self,
        items: List[LexItem],
        n_phonemes: int,
        seed: int = 0,
        max_slots: Optional[int] = None,
    ):
        assert len(items) >= 2

        self.items = items
        self.W = len(items)
        self.S = int(items[0].sem.shape[0])
        self.P = int(n_phonemes)
        self.rng = np.random.default_rng(seed)

        self.word2idx = {it.word: i for i, it in enumerate(items)}

        if max_slots is None:
            max_slots = max(len(it.phon) for it in items)
        self.L = int(max_slots)

        self.END = self.P
        self.P_eff = self.P + 1

        self.M_sem_lex = np.stack([it.sem for it in items], axis=0).astype(float)
        self.M_lex_sem = self.M_sem_lex.T.copy()

        self.M_lex_phon_slots = np.zeros((self.L, self.P_eff, self.W), dtype=float)

        for w, it in enumerate(items):
            for l in range(self.L):
                if l < len(it.phon):
                    p = int(it.phon[l])
                else:
                    p = self.END
                self.M_lex_phon_slots[l, p, w] = 1.0

    def _step(self, a: np.ndarray, net_in: np.ndarray, decay: float, noise_sd: float) -> np.ndarray:
        noise = self.rng.normal(0.0, noise_sd, size=a.shape)
        a_new = (1 - decay) * a + net_in + noise
        return np.clip(a_new, 0.0, None)

    def _softmax(self, x: np.ndarray, temperature: float) -> np.ndarray:
        t = max(temperature, 1e-8)
        z = (x / t) - np.max(x / t)
        ez = np.exp(z)
        return ez / np.sum(ez)

    @staticmethod
    def _jaccard(a: np.ndarray, b: np.ndarray) -> float:
        inter = np.logical_and(a > 0, b > 0).sum()
        union = np.logical_or(a > 0, b > 0).sum()
        return float(inter / union) if union > 0 else 0.0

    @staticmethod
    def _phon_overlap(t: Tuple[int, ...], r: Tuple[int, ...]) -> float:
        ts, rs = set(t), set(r)
        return len(ts & rs) / max(len(ts | rs), 1)

    def _strip_end(self, seq: Tuple[int, ...]) -> Tuple[int, ...]:
        out = []
        for p in seq:
            if p == self.END:
                break
            out.append(p)
        return tuple(out)

    def run_trial(self, target_word: str, p: Params) -> TrialOut:
        tidx = self.word2idx[target_word]
        target = self.items[tidx]

        a_sem = target.sem.astype(float).copy()
        a_lex = np.zeros(self.W, dtype=float)
        a_ph = np.zeros((self.L, self.P_eff), dtype=float)

        target_slot_onehot = self.M_lex_phon_slots[:, :, tidx]

        for step in range(1, p.n_steps + 1):
            fb = np.zeros(self.W, dtype=float)
            for l in range(self.L):
                fb += self.M_lex_phon_slots[l].T @ a_ph[l]

            net_lex = p.gain * (
                p.w_sem_lex * (self.M_sem_lex @ a_sem)
                + p.w_phon_lex * fb
            )

            net_ph = np.zeros_like(a_ph)
            for l in range(self.L):
                net_ph[l] = p.gain * (
                    p.w_lex_phon * (self.M_lex_phon_slots[l] @ a_lex)
                )

            if step == p.jolt_step and p.jolt_mode != "none":
                if p.jolt_mode in ("lex", "both"):
                    net_lex = net_lex.copy()
                    net_lex[tidx] += p.jolt_lex

                if p.jolt_mode in ("phon", "both"):
                    net_ph = net_ph.copy()
                    net_ph += p.jolt_phon * target_slot_onehot

            a_lex = self._step(a_lex, net_lex, p.decay, p.noise_sd)
            a_ph = self._step(a_ph, net_ph, p.decay, p.noise_sd)

            net_sem = p.gain * (p.w_lex_sem * (self.M_lex_sem @ a_lex))
            a_sem = np.clip(target.sem.astype(float) + 0.1 * net_sem, 0.0, None)

        if float(a_lex.max()) < p.omission_thresh_lex:
            return TrialOut(
                target_word=target_word,
                chosen_word=None,
                produced_phon=None,
                error_type="omission",
            )

        probs_lex = self._softmax(a_lex, p.temperature_lex)
        w_hat = int(self.rng.choice(self.W, p=probs_lex))
        chosen_word = self.items[w_hat].word

        produced = []
        for l in range(self.L):
            probs_ph = self._softmax(a_ph[l], p.temperature_phon)
            ph = int(self.rng.choice(self.P_eff, p=probs_ph))
            produced.append(ph)

        produced_seq = self._strip_end(tuple(produced))
        et = self.classify_error(tidx, w_hat, produced_seq)

        return TrialOut(
            target_word=target_word,
            chosen_word=chosen_word,
            produced_phon=produced_seq,
            error_type=et,
        )

    def classify_error(
        self,
        tidx: int,
        w_hat: int,
        produced_seq: Tuple[int, ...],
        sem_thr: float = 0.35,
        phon_thr: float = 0.35,
    ) -> str:
        target = self.items[tidx]
        chosen = self.items[w_hat]

        if w_hat == tidx and produced_seq == target.phon:
            return "correct"

        sem_sim = self._jaccard(target.sem, chosen.sem)
        sem_related = sem_sim >= sem_thr

        phon_sim = self._phon_overlap(target.phon, produced_seq)
        phon_related = phon_sim >= phon_thr

        if w_hat == tidx and produced_seq != target.phon:
            return "phonological"

        if sem_related and phon_related:
            return "mixed"
        if sem_related:
            return "semantic"
        if phon_related:
            return "phonological"
        return "unrelated"

    def simulate_naming(
        self,
        targets: List[str],
        p: Params,
        n_rep: int,
        seed: Optional[int] = None,
    ) -> Dict[str, float]:
        if seed is not None:
            self.rng = np.random.default_rng(seed)

        keys = ["correct", "semantic", "phonological", "mixed", "unrelated", "omission"]
        counts = {k: 0 for k in keys}

        for _ in range(n_rep):
            for tw in targets:
                out = self.run_trial(tw, p)
                counts[out.error_type] += 1

        total = sum(counts.values())
        return {k: counts[k] / total for k in keys}


def make_toy_lexicon() -> Tuple[List[LexItem], int, Dict[str, int]]:
    phoneme_id = {
        "k": 0,
        "æ": 1,
        "t": 2,
        "d": 3,
        "ɔ": 4,
        "g": 5,
        "r": 6,
        "f": 7,
        "m": 8,
    }

    P = len(phoneme_id)

    def sem_vec(*idxs: int) -> np.ndarray:
        v = np.zeros(10, dtype=int)
        v[list(idxs)] = 1
        return v

    def ph(*symbols: str) -> Tuple[int, ...]:
        return tuple(phoneme_id[s] for s in symbols)

    items = [
        LexItem("cat", sem_vec(0, 1, 2), ph("k", "æ", "t")),
        LexItem("dog", sem_vec(0, 1, 3), ph("d", "ɔ", "g")),
        LexItem("rat", sem_vec(0, 2, 4), ph("r", "æ", "t")),
        LexItem("fog", sem_vec(5, 6, 7), ph("f", "ɔ", "g")),
        LexItem("mat", sem_vec(8, 9),    ph("m", "æ", "t")),
    ]

    return items, P, phoneme_id


items, P, phoneme_id = make_toy_lexicon()

model = FoygelDell2000_SlotPhon(
    items=items,
    n_phonemes=P,
    seed=0,
)

targets = [it.word for it in items]

base = Params(
    noise_sd=0.06,
    w_sem_lex=1.2,
    w_lex_sem=1.0,
    w_lex_phon=1.0,
    w_phon_lex=0.7,
    n_steps=16,
    jolt_step=8,
    jolt_mode="both",
    jolt_lex=0.4,
    jolt_phon=0.4,
    temperature_lex=0.7,
    temperature_phon=0.7,
    omission_thresh_lex=0.05,
)

dist = model.simulate_naming(
    targets=targets,
    p=base,
    n_rep=200,
    seed=42,
)

print("phoneme_id:", phoneme_id)
print("targets:", targets)
print("error distribution:", dist)

In [ ]:
items, P, phoneme_id = make_toy_lexicon()
for _item in items:
    print(_item)

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, asdict
import numpy as np
from typing import Dict, List, Tuple, Optional, Literal

JoltMode = Literal["lex", "phon", "both", "none"]

# ============================================================
# 0) Slot-based simulator (the shared generative model)
# ============================================================

@dataclass(frozen=True)
class LexItem:
    word: str
    sem: np.ndarray
    phon: Tuple[int, ...]

@dataclass(frozen=True)
class Params:
    # connection strengths
    w_sem_lex: float = 1.0
    w_lex_sem: float = 1.0
    w_lex_phon: float = 1.0
    w_phon_lex: float = 1.0

    # dynamics
    decay: float = 0.25
    gain: float = 1.0
    noise_sd: float = 0.05

    # timing
    n_steps: int = 16
    jolt_step: int = 8

    # jolt
    jolt_mode: JoltMode = "both"
    jolt_lex: float = 0.4
    jolt_phon: float = 0.4

    # selection temps
    temperature_lex: float = 0.7
    temperature_phon: float = 0.7

    # omission
    omission_thresh_lex: float = 0.05


class FoygelDell2000_SlotPhon:
    def __init__(self, items: List[LexItem], n_phonemes: int, seed: int = 0, max_slots: Optional[int] = None):
        self.items = items
        self.W = len(items)
        self.S = int(items[0].sem.shape[0])
        self.P = int(n_phonemes)
        self.rng = np.random.default_rng(seed)
        self.word2idx = {it.word: i for i, it in enumerate(items)}

        if max_slots is None:
            max_slots = max(len(it.phon) for it in items)
        self.L = int(max_slots)

        # END symbol
        self.END = self.P
        self.P_eff = self.P + 1

        # Sem <-> Lex
        self.M_sem_lex = np.stack([it.sem for it in items], axis=0).astype(float)  # (W,S)
        self.M_lex_sem = self.M_sem_lex.T.copy()                                   # (S,W)

        # Lex -> PhonSlots: (L, P_eff, W) one-hot per slot
        self.M_lex_phon_slots = np.zeros((self.L, self.P_eff, self.W), dtype=float)
        for w, it in enumerate(items):
            for l in range(self.L):
                ph = int(it.phon[l]) if l < len(it.phon) else self.END
                self.M_lex_phon_slots[l, ph, w] = 1.0

    def _step(self, a: np.ndarray, net_in: np.ndarray, decay: float, noise_sd: float) -> np.ndarray:
        noise = self.rng.normal(0.0, noise_sd, size=a.shape)
        return np.clip((1 - decay) * a + net_in + noise, 0.0, None)

    def _softmax(self, x: np.ndarray, temperature: float) -> np.ndarray:
        t = max(temperature, 1e-8)
        z = (x / t) - np.max(x / t)
        ez = np.exp(z)
        return ez / np.sum(ez)

    def _strip_end(self, seq: Tuple[int, ...]) -> Tuple[int, ...]:
        out = []
        for p in seq:
            if p == self.END:
                break
            out.append(p)
        return tuple(out)

    @staticmethod
    def _jaccard(a: np.ndarray, b: np.ndarray) -> float:
        inter = np.logical_and(a > 0, b > 0).sum()
        union = np.logical_or(a > 0, b > 0).sum()
        return float(inter / union) if union > 0 else 0.0

    @staticmethod
    def _phon_overlap(t: Tuple[int, ...], r: Tuple[int, ...]) -> float:
        ts, rs = set(t), set(r)
        return len(ts & rs) / max(len(ts | rs), 1)

    def classify_error(self, tidx: int, w_hat: int, produced_seq: Tuple[int, ...],
                       sem_thr: float = 0.35, phon_thr: float = 0.35) -> str:
        target = self.items[tidx]
        chosen = self.items[w_hat]

        if w_hat == tidx and produced_seq == target.phon:
            return "correct"

        sem_sim = self._jaccard(target.sem, chosen.sem)
        sem_related = sem_sim >= sem_thr

        phon_sim = self._phon_overlap(target.phon, produced_seq)
        phon_related = phon_sim >= phon_thr

        if w_hat == tidx and produced_seq != target.phon:
            return "phonological"

        if sem_related and phon_related:
            return "mixed"
        if sem_related:
            return "semantic"
        if phon_related:
            return "phonological"
        return "unrelated"

    def run_trial(self, target_word: str, p: Params) -> Tuple[int, Optional[int], Optional[Tuple[int, ...]], str]:
        tidx = self.word2idx[target_word]
        target = self.items[tidx]

        a_sem = target.sem.astype(float).copy()
        a_lex = np.zeros(self.W, dtype=float)
        a_ph  = np.zeros((self.L, self.P_eff), dtype=float)

        target_slot_onehot = self.M_lex_phon_slots[:, :, tidx]  # (L, P_eff)

        for step in range(1, p.n_steps + 1):
            # phon->lex feedback
            fb = np.zeros(self.W, dtype=float)
            for l in range(self.L):
                fb += self.M_lex_phon_slots[l].T @ a_ph[l]

            net_lex = p.gain * (p.w_sem_lex * (self.M_sem_lex @ a_sem) + p.w_phon_lex * fb)

            net_ph = np.zeros_like(a_ph)
            for l in range(self.L):
                net_ph[l] = p.gain * (p.w_lex_phon * (self.M_lex_phon_slots[l] @ a_lex))

            # jolt
            if step == p.jolt_step and p.jolt_mode != "none":
                if p.jolt_mode in ("lex", "both"):
                    net_lex = net_lex.copy()
                    net_lex[tidx] += p.jolt_lex
                if p.jolt_mode in ("phon", "both"):
                    net_ph = net_ph.copy()
                    net_ph += p.jolt_phon * target_slot_onehot

            a_lex = self._step(a_lex, net_lex, p.decay, p.noise_sd)
            a_ph  = self._step(a_ph,  net_ph,  p.decay, p.noise_sd)

            net_sem = p.gain * (p.w_lex_sem * (self.M_lex_sem @ a_lex))
            a_sem = np.clip(target.sem.astype(float) + 0.1 * net_sem, 0.0, None)

        # omission
        if float(a_lex.max()) < p.omission_thresh_lex:
            return tidx, None, None, "omission"

        # lexical choice
        probs_lex = self._softmax(a_lex, p.temperature_lex)
        w_hat = int(self.rng.choice(self.W, p=probs_lex))

        # slot phon production
        produced = []
        for l in range(self.L):
            probs_ph = self._softmax(a_ph[l], p.temperature_phon)
            ph = int(self.rng.choice(self.P_eff, p=probs_ph))
            produced.append(ph)
        produced_seq = self._strip_end(tuple(produced))

        et = self.classify_error(tidx, w_hat, produced_seq)
        return tidx, w_hat, produced_seq, et

    def simulate_naming(self, targets: List[str], p: Params, n_rep: int,
                        seed: Optional[int] = None,
                        return_confusion: bool = True) -> Tuple[Dict[str, float], Optional[np.ndarray], int]:
        if seed is not None:
            self.rng = np.random.default_rng(seed)

        keys = ["correct","semantic","phonological","mixed","unrelated","omission"]
        counts = {k: 0 for k in keys}
        conf = np.zeros((self.W, self.W), dtype=int) if return_confusion else None

        for _ in range(n_rep):
            for tw in targets:
                tidx, w_hat, _, et = self.run_trial(tw, p)
                counts[et] += 1
                if conf is not None and w_hat is not None:
                    conf[tidx, w_hat] += 1

        total = sum(counts.values())
        rates = {k: counts[k]/total for k in keys}
        return rates, conf, total


# ============================================================
# 1) Observations and scoring (rates + confusion)
# ============================================================

@dataclass(frozen=True)
class Observed:
    rates: Dict[str, float]
    confusion: np.ndarray          # (W,W) counts
    total_trials: int              # totals behind rates/confusion

def normalize_conf(conf: np.ndarray, eps: float = 1e-9) -> np.ndarray:
    # row-normalize to remove target frequency effects
    row = conf.sum(axis=1, keepdims=True)
    return conf / np.maximum(row, eps)

def score_rates_sse(sim_rates: Dict[str, float], obs_rates: Dict[str, float]) -> float:
    keys = sorted(set(sim_rates.keys()) | set(obs_rates.keys()))
    return float(sum((sim_rates.get(k,0.0) - obs_rates.get(k,0.0))**2 for k in keys))

def score_confusion_fro(sim_conf: np.ndarray, obs_conf: np.ndarray) -> float:
    A = normalize_conf(sim_conf)
    B = normalize_conf(obs_conf)
    return float(np.mean((A - B)**2))

def combined_score(sim_rates, sim_conf, obs: Observed, alpha_rates=1.0, beta_conf=1.0) -> float:
    # Combine two pieces of naming-only evidence
    r = score_rates_sse(sim_rates, obs.rates)
    c = score_confusion_fro(sim_conf, obs.confusion)
    return float(alpha_rates * r + beta_conf * c)


# ============================================================
# 2) Two parameterizations: WD vs SP
# ============================================================

@dataclass(frozen=True)
class WDParams:
    """
    Weight-Decay parameterization (Dell et al 1997 style):
      - global weight scaling (w)
      - global decay (d)
    plus optional: noise and jolt (kept shared to be fair)
    """
    w_global: float
    decay: float
    noise_sd: float
    jolt_mode: JoltMode
    jolt_lex: float
    jolt_phon: float

@dataclass(frozen=True)
class SPParams:
    """
    Semantic-Phonological parameterization (Foygel & Dell 2000 style):
      - s = semantic weight (Sem<->Lex)
      - p = phonological weight (Lex<->Phon and Phon->Lex)
    plus optional: noise and jolt (shared knobs)
    """
    s_weight: float
    p_weight: float
    decay: float
    noise_sd: float
    jolt_mode: JoltMode
    jolt_lex: float
    jolt_phon: float

def wd_to_params(base: Params, wd: WDParams) -> Params:
    # global scaling applies to all major connections
    return Params(**{
        **asdict(base),
        "w_sem_lex": base.w_sem_lex * wd.w_global,
        "w_lex_sem": base.w_lex_sem * wd.w_global,
        "w_lex_phon": base.w_lex_phon * wd.w_global,
        "w_phon_lex": base.w_phon_lex * wd.w_global,
        "decay": wd.decay,
        "noise_sd": wd.noise_sd,
        "jolt_mode": wd.jolt_mode,
        "jolt_lex": wd.jolt_lex,
        "jolt_phon": wd.jolt_phon,
    })

def sp_to_params(base: Params, sp: SPParams) -> Params:
    # semantic path vs phonological path
    return Params(**{
        **asdict(base),
        "w_sem_lex": base.w_sem_lex * sp.s_weight,
        "w_lex_sem": base.w_lex_sem * sp.s_weight,
        "w_lex_phon": base.w_lex_phon * sp.p_weight,
        "w_phon_lex": base.w_phon_lex * sp.p_weight,
        "decay": sp.decay,
        "noise_sd": sp.noise_sd,
        "jolt_mode": sp.jolt_mode,
        "jolt_lex": sp.jolt_lex,
        "jolt_phon": sp.jolt_phon,
    })


# ============================================================
# 3) Grid search that RETURNS MANY near-optimal fits
# ============================================================

@dataclass(frozen=True)
class FitRow:
    kind: str          # "WD" or "SP"
    raw_params: object # WDParams or SPParams
    score: float
    sim_rates: Dict[str, float]

def grid_search_wd(
    model: FoygelDell2000_SlotPhon,
    targets: List[str],
    obs: Observed,
    base: Params,
    grid_w: np.ndarray,
    grid_decay: np.ndarray,
    grid_noise: np.ndarray,
    grid_jlex: np.ndarray,
    grid_jphon: np.ndarray,
    jolt_modes: List[JoltMode],
    n_rep: int = 200,
    alpha_rates: float = 1.0,
    beta_conf: float = 1.0,
    top_n: int = 50,
    seed0: int = 1000,
) -> List[FitRow]:
    rows: List[FitRow] = []
    cid = 0
    for jm in jolt_modes:
        for w in grid_w:
            for d in grid_decay:
                for ns in grid_noise:
                    for jlex in grid_jlex:
                        for jph in grid_jphon:
                            wd = WDParams(float(w), float(d), float(ns), jm, float(jlex), float(jph))
                            p = wd_to_params(base, wd)
                            sim_rates, sim_conf, _ = model.simulate_naming(targets, p, n_rep=n_rep, seed=seed0+cid, return_confusion=True)
                            cid += 1
                            sc = combined_score(sim_rates, sim_conf, obs, alpha_rates, beta_conf)
                            rows.append(FitRow("WD", wd, sc, sim_rates))
    rows.sort(key=lambda r: r.score)
    return rows[:top_n]

def grid_search_sp(
    model: FoygelDell2000_SlotPhon,
    targets: List[str],
    obs: Observed,
    base: Params,
    grid_s: np.ndarray,
    grid_p: np.ndarray,
    grid_decay: np.ndarray,
    grid_noise: np.ndarray,
    grid_jlex: np.ndarray,
    grid_jphon: np.ndarray,
    jolt_modes: List[JoltMode],
    n_rep: int = 200,
    alpha_rates: float = 1.0,
    beta_conf: float = 1.0,
    top_n: int = 50,
    seed0: int = 2000,
) -> List[FitRow]:
    rows: List[FitRow] = []
    cid = 0
    for jm in jolt_modes:
        for s in grid_s:
            for pwt in grid_p:
                for d in grid_decay:
                    for ns in grid_noise:
                        for jlex in grid_jlex:
                            for jph in grid_jphon:
                                sp = SPParams(float(s), float(pwt), float(d), float(ns), jm, float(jlex), float(jph))
                                p = sp_to_params(base, sp)
                                sim_rates, sim_conf, _ = model.simulate_naming(targets, p, n_rep=n_rep, seed=seed0+cid, return_confusion=True)
                                cid += 1
                                sc = combined_score(sim_rates, sim_conf, obs, alpha_rates, beta_conf)
                                rows.append(FitRow("SP", sp, sc, sim_rates))
    rows.sort(key=lambda r: r.score)
    return rows[:top_n]


# ============================================================
# 4) Identifiability diagnostics: “how many different causes fit?”
# ============================================================

def param_spread(rows: List[FitRow]) -> Dict[str, Tuple[float, float]]:
    """
    Min-max spread over the top fits for numeric fields.
    """
    if not rows:
        return {}
    # collect dicts
    dlist = []
    for r in rows:
        d = asdict(r.raw_params)  # works for dataclasses
        dlist.append(d)
    keys = dlist[0].keys()
    out = {}
    for k in keys:
        vals = [d[k] for d in dlist]
        if all(isinstance(v, (int, float, np.floating)) for v in vals):
            out[k] = (float(np.min(vals)), float(np.max(vals)))
    return out

def count_distinct_modes(rows: List[FitRow]) -> Dict[str, int]:
    d = {}
    for r in rows:
        jm = asdict(r.raw_params)["jolt_mode"]
        d[jm] = d.get(jm, 0) + 1
    return d

def pretty_top(rows: List[FitRow], n: int = 10):
    for r in rows[:n]:
        print(f"[{r.kind}] score={r.score:.6g} params={r.raw_params} rates={r.sim_rates}")

# ============================================================
# 5) Toy lexicon + a “synthetic patient” observation
# ============================================================

def make_toy_lexicon() -> Tuple[List[LexItem], int]:
    P = 12
    def sem_vec(*idxs: int) -> np.ndarray:
        v = np.zeros(10, dtype=int); v[list(idxs)] = 1; return v
    items = [
        LexItem("cat", sem_vec(0,1,2), (0,1,2)),
        LexItem("dog", sem_vec(0,1,3), (0,4,2)),
        LexItem("rat", sem_vec(0,2,4), (5,1,2)),
        LexItem("car", sem_vec(5,6),   (5,7,8)),
        LexItem("bus", sem_vec(5,7),   (9,7,10)),
        LexItem("cup", sem_vec(8,9),   (0,7,11)),
    ]
    return items, P

def make_synthetic_observation(model, targets, base: Params, true_kind: str = "SP", seed: int = 42) -> Observed:
    """
    Creates a synthetic “patient” so you can test identifiability cleanly.
    Replace this with real patient data when ready.
    """
    if true_kind == "SP":
        true = SPParams(s_weight=0.8, p_weight=1.1, decay=0.30, noise_sd=0.08,
                        jolt_mode="both", jolt_lex=0.3, jolt_phon=0.4)
        p = sp_to_params(base, true)
    else:
        true = WDParams(w_global=0.85, decay=0.33, noise_sd=0.08,
                        jolt_mode="lex", jolt_lex=0.5, jolt_phon=0.0)
        p = wd_to_params(base, true)

    rates, conf, total = model.simulate_naming(targets, p, n_rep=400, seed=seed, return_confusion=True)
    assert conf is not None
    return Observed(rates=rates, confusion=conf, total_trials=total)

# ============================================================
# 6) Main
# ============================================================

if __name__ == "__main__":
    items, P = make_toy_lexicon()
    model = FoygelDell2000_SlotPhon(items, n_phonemes=P, seed=0)
    targets = [it.word for it in items]

    base = Params(
        n_steps=16, jolt_step=8,
        # base connection scales (1.0 means raw parameterization controls it)
        w_sem_lex=1.0, w_lex_sem=1.0, w_lex_phon=1.0, w_phon_lex=1.0,
        temperature_lex=0.7, temperature_phon=0.7,
        omission_thresh_lex=0.05,
        jolt_mode="both", jolt_lex=0.4, jolt_phon=0.4,
        noise_sd=0.06, decay=0.25
    )

    # --- Observed data (synthetic; replace with your patient naming data) ---
    obs = make_synthetic_observation(model, targets, base, true_kind="SP", seed=123)
    print("Observed (synthetic) rates:", obs.rates)
    print("Observed confusion row-normalized (first row):", normalize_conf(obs.confusion)[0])

    # --- grids (keep modest first; then enlarge) ---
    jolt_modes = ["lex", "phon", "both"]

    grid_decay = np.linspace(0.20, 0.45, 6)
    grid_noise = np.linspace(0.02, 0.12, 6)
    grid_jlex  = np.linspace(0.0, 0.8, 5)
    grid_jphon = np.linspace(0.0, 0.8, 5)

    # WD grids
    grid_w = np.linspace(0.5, 1.4, 10)

    # SP grids
    grid_s = np.linspace(0.5, 1.5, 11)
    grid_p = np.linspace(0.5, 1.5, 11)

    # weights for the combined score
    # beta_conf を上げると「混同行列」をより重視（同定が少し改善することも）
    alpha_rates = 1.0
    beta_conf   = 2.0

    # --- Fit WD and SP under the SAME observation ---
    wd_top = grid_search_wd(
        model, targets, obs, base,
        grid_w, grid_decay, grid_noise, grid_jlex, grid_jphon, jolt_modes,
        n_rep=250, alpha_rates=alpha_rates, beta_conf=beta_conf, top_n=40, seed0=10000
    )
    sp_top = grid_search_sp(
        model, targets, obs, base,
        grid_s, grid_p, grid_decay, grid_noise, grid_jlex, grid_jphon, jolt_modes,
        n_rep=250, alpha_rates=alpha_rates, beta_conf=beta_conf, top_n=40, seed0=20000
    )

    print("\n=== TOP fits: WD ===")
    pretty_top(wd_top, n=8)
    print("WD spread (top fits):", param_spread(wd_top))
    print("WD jolt_mode counts:", count_distinct_modes(wd_top))

    print("\n=== TOP fits: SP ===")
    pretty_top(sp_top, n=8)
    print("SP spread (top fits):", param_spread(sp_top))
    print("SP jolt_mode counts:", count_distinct_modes(sp_top))

    # --- Compare “how non-identifiable” they look ---
    print("\n=== Quick comparison ===")
    print(f"Best WD score: {wd_top[0].score:.6g}")
    print(f"Best SP score: {sp_top[0].score:.6g}")

Observed (synthetic) rates: {'correct': 0.16666666666666666, 'semantic': 0.0, 'phonological': 0.0, 'mixed': 0.3333333333333333, 'unrelated': 0.5, 'omission': 0.0}
Observed confusion row-normalized (first row): [1. 0. 0. 0. 0. 0.]
